In [1]:
try:
    %pip install --user "oracledb" --no-warn-script-location
except Exception as e:
    print("\x1b[31m\u2717 Unexpected error! Please contact course staff\n" +
         "Please include the entire text above and below in your message.")
    raise

Note: you may need to restart the kernel to use updated packages.


In [2]:
import oracledb
dsn = oracledb.makedsn("localhost", 1522, service_name="stu")
connection = oracledb.connect(user="ora_ahasjim", password="a31495526", dsn=dsn)

In [3]:
# Importing packages
import pandas as pd
import numpy as np

rt_movies = pd.read_csv("datasets/rotten_tomatoes_movies.csv")

In [4]:
rt_movies

,id,title,audienceScore,tomatoMeter,rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
0,space-zombie-bingo,Space Zombie Bingo!,50.0,NaN,NaN,NaN,NaN,2018-08-25,75.0,"Comedy, Horror, Sci-fi",English,George Ormrod,"George Ormrod,John Sabotta",NaN,NaN,NaN
1,the_green_grass,The Green Grass,NaN,NaN,NaN,NaN,NaN,2020-02-11,114.0,Drama,English,Tiffany Edwards,Tiffany Edwards,NaN,NaN,NaN
2,love_lies,"Love, Lies",43.0,NaN,NaN,NaN,NaN,NaN,120.0,Drama,Korean,"Park Heung-Sik,Heung-Sik Park","Ha Young-Joon,Jeon Yun-su,Song Hye-jin",NaN,NaN,NaN
3,the_sore_losers_1997,Sore Losers,60.0,NaN,NaN,NaN,NaN,2020-10-23,90.0,"Action, Mystery & thriller",English,John Michael McCarthy,John Michael McCarthy,NaN,NaN,NaN
4,dinosaur_island_2002,Dinosaur Island,70.0,NaN,NaN,NaN,NaN,2017-03-27,80.0,"Fantasy, Adventure, Animation",English,Will Meugniot,John Loy,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143253,nadia_the_secret_of_blue_water_the_motion_pict...,Nadia: The Secret of Blue Water: The Motion Pi...,14.0,NaN,NaN,NaN,2002-08-27,NaN,90.0,"Action, Adventure, Anime",Japanese,Sho Aono,Kaoru Umeno,NaN,ADV Films,NaN
143254,everyone_i_knew_and_loved,Everyone I Knew and Loved,NaN,NaN,NaN,NaN,NaN,NaN,99.0,Drama,English,Andrew Behringer,Erika Heidewald,NaN,NaN,NaN
143255,the-human-body,The Human Body,71.0,89.0,NaN,NaN,NaN,NaN,43.0,Documentary,English,Peter Georgi,Richard Dale,NaN,NaN,NaN
143256,flying_fists,Flying Fists,NaN,NaN,NaN,NaN,NaN,2006-11-21,63.0,Drama,English,Robert F. Hill,"Robert F. Hill,Basil Dickey",NaN,NaN,NaN


In [5]:
# Connecting to the server
cur = connection.cursor()

In [6]:
cur.execute("DROP TABLE rt_movies")
connection.commit()

cur.execute("PURGE RECYCLEBIN")
connection.commit()

In [7]:
create_table = """
CREATE TABLE rt_movies (
    id VARCHAR2(1000) PRIMARY KEY,
    title VARCHAR2(255),
    audienceScore NUMBER,
    tomatoMeter NUMBER,
    rating VARCHAR2(20),
    ratingContents VARCHAR2(500),
    releaseDateTheaters DATE,
    releaseDateStreaming DATE,
    runtimeMinutes NUMBER,
    genre VARCHAR2(255),
    originalLanguage VARCHAR2(50),
    director VARCHAR2(500),
    writer VARCHAR2(500),
    boxOffice NUMBER,
    distributor VARCHAR2(255),
    soundMix VARCHAR2(255)
)
"""

cur.execute(create_table)
connection.commit()

In [8]:
import pandas as pd
import numpy as np

# load dataset
rt_movies = pd.read_csv("datasets/rotten_tomatoes_movies.csv")

# remove duplicates
rt_movies = rt_movies.drop_duplicates(subset=["id"])

# replace NaN with None so Oracle sees them as NULL
rt_movies = rt_movies.replace({np.nan: None})
rt_movies = rt_movies.where(pd.notnull(rt_movies), None)

# convert date columns
rt_movies["releaseDateTheaters"] = pd.to_datetime(
    rt_movies["releaseDateTheaters"], errors="coerce"
)

rt_movies["releaseDateStreaming"] = pd.to_datetime(
    rt_movies["releaseDateStreaming"], errors="coerce"
)

# convert numeric columns
rt_movies["audienceScore"] = pd.to_numeric(rt_movies["audienceScore"], errors="coerce")
rt_movies["tomatoMeter"] = pd.to_numeric(rt_movies["tomatoMeter"], errors="coerce")
rt_movies["runtimeMinutes"] = pd.to_numeric(rt_movies["runtimeMinutes"], errors="coerce")

# Clean boxOffice first, then convert
rt_movies["boxOffice"] = (
    rt_movies["boxOffice"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

rt_movies["boxOffice"] = pd.to_numeric(rt_movies["boxOffice"], errors="coerce")

# replace NaN again after conversions
rt_movies = rt_movies.where(pd.notnull(rt_movies), None)

In [9]:
insert_sql = """
INSERT INTO rt_movies (
    id, title, audienceScore, tomatoMeter, rating, ratingContents,
    releaseDateTheaters, releaseDateStreaming, runtimeMinutes, genre,
    originalLanguage, director, writer, boxOffice, distributor, soundMix
) VALUES (
    :1, :2, :3, :4, :5, :6,
    :7, :8, :9, :10,
    :11, :12, :13, :14, :15, :16
)
"""

In [10]:
rows_to_insert = []

for _, row in rt_movies.iterrows():
    rows_to_insert.append((
        row["id"],
        row["title"],
        float(row["audienceScore"]) if pd.notna(row["audienceScore"]) else None,
        float(row["tomatoMeter"]) if pd.notna(row["tomatoMeter"]) else None,
        row["rating"],
        row["ratingContents"],
        row["releaseDateTheaters"].to_pydatetime() if pd.notna(row["releaseDateTheaters"]) else None,
        row["releaseDateStreaming"].to_pydatetime() if pd.notna(row["releaseDateStreaming"]) else None,
        int(row["runtimeMinutes"]) if pd.notna(row["runtimeMinutes"]) else None,
        row["genre"],
        row["originalLanguage"],
        row["director"],
        row["writer"],
        float(row["boxOffice"]) if pd.notna(row["boxOffice"]) else None,
        row["distributor"],
        row["soundMix"]
    ))

In [11]:
cur.executemany(insert_sql, rows_to_insert)
connection.commit()

cur.execute("SELECT COUNT(*) FROM rt_movies")
print(cur.fetchone())

DatabaseError: ORA-01536: space quota exceeded for tablespace 'USERS'
Help: https://docs.oracle.com/error-help/db/ora-01536/